> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# 123 de compra: backtest and optimization

One ticker, one strategy, from candles to charts. The rules live in `docs/brief.md`, section 7 (the owner's pseudocode); this notebook only sets parameters and calls `b3core` (engine, metrics, optimizer) and `b3report` (charts). It holds no trading logic.

**Before trusting any number here:**
- Candle times are assumed to be `America/Sao_Paulo` wall-clock time, and it is not known whether the MT5 history is adjusted for dividends and splits (`docs/data-notes.md`).
- Several engine details are assumptions waiting for sign-off (brief, section 7).
- The first and last candle of each day differ slightly from QuantBrasil's data (section 6 below).
- `score_v1` is not implemented yet: its formula needs approval first, so the objectives are result, profit factor, EV and drawdown.
- 123 de venda is not specified yet.

In [ ]:
from collections import Counter
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from b3core.data import CsvProvider, Timeframe
from b3core.engine import BacktestConfig, run_backtest
from b3core.optimizer import (
    Constraints,
    GridSearch,
    ParamRange,
    ParamSpace,
    RandomSearch,
    describe_space,
    optimize,
)
from b3report.charts import (
    equity_curve_figure,
    heatmap_figure,
    monthly_returns,
    monthly_returns_figure,
    trade_chart,
)

pd.options.display.float_format = "{:,.4f}".format

## 1. Parameters

Everything the run depends on is a plain variable here, with the pseudocode's names and defaults. The CSV folder is gitignored vendor data: never commit it.

In [ ]:
DATA_DIR = Path("../samples")  # MT5 "export bars" files, gitignored
SYMBOL = "PETR4"
TIMEFRAME = Timeframe.H2  # D1, H2 or H4 with the current samples
START = datetime(2016, 1, 1, tzinfo=UTC)
END = datetime(2027, 1, 1, tzinfo=UTC)  # exclusive

CONFIG = BacktestConfig(
    capital_brl=100_000,
    max_risk_brl=2_000,
    tick_size=0.01,
    lot_size=100,
    ma_fast_period=8,  # grid 5..13
    ma_slow_period=80,  # grid 50..200
    use_ma_order=True,  # fast above slow
    use_fast_slope=True,  # fast rising
    use_slow_slope=True,  # slow rising
    use_price_above_fast=False,  # three closes above fast
    max_c3_ticks=0,  # 0 = off
    entry_offset_ticks=1,  # grid 0..2
    entry_validity_bars=1,
    stop_mode="C3",  # "C3" or "C2"
    stop_offset_ticks=1,
    target_mode="AMP",  # "AMP" (amplitude) or "RISK"
    target_mult=1.61,  # AMP: 1, 1.272, 1.61, 2   RISK: 1, 1.5, 2, 3
)

## 2. Load candles

Problems found on ingest are listed, never silently fixed (brief, section 5).

In [ ]:
candles = CsvProvider(DATA_DIR).get_candles(SYMBOL, TIMEFRAME, START, END)
print(f"{SYMBOL} {TIMEFRAME.value}: {len(candles)} candles, {candles.ts.iloc[0]} to {candles.ts.iloc[-1]}")
print("ingest issues:", candles.attrs["issues"] or "none")
candles.head()

## 3. Check the engine against the worked example

The five candles of brief section 7, with the trend checks off, a `C3` stop, a `RISK` target times 2 and a max risk of R$ 200. Worked out by hand: entry 10.41, stop 10.24, risk 0.17, 1,100 shares, target 10.75, so **+R$ 374.00**, exactly 2.0 R.

In [ ]:
example = pd.DataFrame(
    {
        "ts": pd.date_range("2024-01-02 13:00", periods=5, freq="2h", tz="UTC"),
        "open": [10.50, 10.45, 10.30, 10.38, 10.44],
        "high": [10.60, 10.50, 10.40, 10.45, 10.80],
        "low": [10.40, 10.20, 10.25, 10.35, 10.40],
        "close": [10.45, 10.30, 10.38, 10.44, 10.78],
        "volume": 1000.0,
    }
)
EXAMPLE_CONFIG = BacktestConfig(
    max_risk_brl=200,
    use_ma_order=False,
    use_fast_slope=False,
    use_slow_slope=False,
    target_mode="RISK",
    target_mult=2.0,
)
(trade,) = run_backtest(example, EXAMPLE_CONFIG).trades
print(f"entry {trade.entry:.2f}  stop {trade.stop:.2f}  target {trade.target:.2f}  qty {trade.qty}  pnl R$ {trade.pnl:.2f}  ({trade.r_multiple:.1f} R)")
assert round(trade.pnl, 2) == 374.00 and trade.qty == 1100, "worked example failed"
trade_chart(example, trade, context=3)

## 4. Backtest

In [ ]:
result = run_backtest(candles, CONFIG)
pd.Series(result.metrics.to_dict(), name=f"{SYMBOL} {TIMEFRAME.value}").to_frame()

In [ ]:
trades = pd.DataFrame([vars(t) for t in result.trades])
trades.head(10)

In [ ]:
equity_curve_figure(result)

In [ ]:
monthly_returns_figure(result)

The best and the worst trade on the candle chart (stop and target as dashed lines).

In [ ]:
if result.trades:
    best = max(result.trades, key=lambda t: t.pnl)
    worst = min(result.trades, key=lambda t: t.pnl)
    display(trade_chart(candles, best), trade_chart(candles, worst))
else:
    print("no trades with these parameters")

## 5. Optimization

Each parameter is a fixed value, a `ParamRange(start, stop, step)` or a list of options. The optimizer reports the size of the space before running, searches **in-sample only**, then re-runs the best results **out-of-sample** and flags large drops (brief, section 12). The out-of-sample slice restarts its moving averages, so it loses its first bars to their warm-up.

Read the answer with care: the best of many trials is partly luck, so always look at `n_trials` next to it, at the out-of-sample columns and at the heatmap.

In [ ]:
SPACE = ParamSpace(
    {
        "ma_fast_period": ParamRange(5, 13, 4),  # 5, 9, 13
        "ma_slow_period": ParamRange(50, 200, 75),  # 50, 125, 200
        "entry_offset_ticks": ParamRange(0, 2, 1),
        "stop_mode": ["C3", "C2"],
        "target_mode": ["AMP", "RISK"],
        "target_mult": [1.0, 1.5, 2.0, 3.0],
    }
)
info = describe_space(SPACE)
print(f"search space: {info.size} combinations, recommended algorithm: {info.recommendation}")

In [ ]:
OBJECTIVE = "result"  # "result", "profit_factor", "ev" or "min_drawdown"
CONSTRAINTS = Constraints(min_trades=30)  # also: max_drawdown=0.25
ALGORITHM = GridSearch()  # or RandomSearch(seed=42, n_samples=100)
WORKERS = 1  # >1 evaluates trials in a process pool, same results

opt = optimize(
    candles,
    CONFIG,
    SPACE,
    objective=OBJECTIVE,
    constraints=CONSTRAINTS,
    algorithm=ALGORITHM,
    oos_fraction=0.3,  # last 30% of the candles are never used by the search
    top_n=5,
    workers=WORKERS,
)
best = opt.best
print(f"{opt.n_trials} trials. best in-sample {OBJECTIVE}: {best.objective:+.4f}")
print("best parameters:", best.params)

In-sample versus out-of-sample for the top results. `dropped` is True when the out-of-sample objective lost more than half of the in-sample one.

In [ ]:
opt.validation_table()

Top of the in-sample ranking (infeasible trials, those breaking the constraints, are ranked last).

In [ ]:
trials = opt.trials_table()
ranked = trials.sort_values(["feasible", "objective"], ascending=False)
ranked.drop(columns=["run_hash", "phase"]).head(10)

**Neighbourhood view.** A good point surrounded by bad neighbours is a warning sign; a broad plateau is a good sign. Cells average the parameters not shown.

In [ ]:
heatmap_figure(trials, x="ma_slow_period", y="ma_fast_period")

In [ ]:
heatmap_figure(trials, x="target_mult", y="target_mode")

## 6. Day trade, and a check against QuantBrasil

Nothing is held overnight, but an order from the last candle of a day may fill on the next morning. `min_entry_time` and `max_entry_time` bound the candle that fills an entry (defaults: start of day, and the exit time), and `exit_time` (here 16:30) is the forced exit. Day trade needs H2 or finer: on H4 only a handful of candles can qualify as entries.

**How the forced exit is priced** (`day_exit_policy`): `last_candle_close` (the default, the QuantBrasil behaviour) leaves at the close of the candle that contains 16:30; `auto` pinpoints 16:30 with finer candles when an M30 or M15 export is loaded below; `last_candle_open` leaves at that candle's open.

In [ ]:
DAY_CONFIG = replace(
    CONFIG,
    day_trade=True,
    min_entry_time="00:00",  # start of day
    max_entry_time=None,  # None means exit_time
    exit_time="16:30",  # end-of-day call
    day_exit_policy="last_candle_close",  # or "auto" (needs INTRADAY) or "last_candle_open"
)

try:
    INTRADAY = CsvProvider(DATA_DIR).get_candles(SYMBOL, Timeframe.M30, START, END)
    print(f"finer candles: {len(INTRADAY)} M30 candles, used by day_exit_policy='auto'")
except FileNotFoundError:
    INTRADAY = None
    print("no M30 file for this symbol: only the close and open policies are available")

day_result = run_backtest(candles, DAY_CONFIG, timeframe=TIMEFRAME, intraday=INTRADAY)
print("exit reasons:", dict(Counter(t.exit_reason for t in day_result.trades)))
pd.Series(day_result.metrics.to_dict(), name="day trade").to_frame()

Three real QuantBrasil trades on PETR4 H2 (screenshots, copied here by hand) next to ours. Trade 1 should match exactly. Trade 3 is the same trade with slightly different prices: the first and the last candle of the day differ between MT5 and QuantBrasil (`docs/data-notes.md`). Trade 2 does not appear in our run: it needs a valid pattern on the 04/09 candles that our data does not have.

In [ ]:
quantbrasil = pd.DataFrame(
    [
        ("2026-09-02", 47.95, 48.20, 2000, 500.0),
        ("2026-09-08", 48.10, 48.09, 1700, -17.0),
        ("2026-09-09", 48.80, 48.45, 2000, -700.0),
    ],
    columns=["date", "entry", "exit", "qty", "pnl"],
)
ours = pd.DataFrame(
    [
        (t.entry_ts.tz_convert("America/Sao_Paulo").strftime("%Y-%m-%d"), t.entry, t.exit, t.qty, round(t.pnl, 2))
        for t in day_result.trades
        if "2026-09-01" <= t.entry_ts.tz_convert("America/Sao_Paulo").strftime("%Y-%m-%d") <= "2026-09-11"
    ],
    columns=["date", "entry", "exit", "qty", "pnl"],
)
display(quantbrasil.assign(source="QuantBrasil"), ours.assign(source="ours"))

In [ ]:
display(equity_curve_figure(day_result))
forced = [t for t in day_result.trades if t.exit_reason == "time"]
if forced:
    display(trade_chart(candles, forced[-1]))

Optimization of the entry window. Combinations whose window is invalid (minimum not before the maximum, or a maximum after the exit time) are dropped before any backtest runs.

In [ ]:
DAY_SPACE = ParamSpace(
    {
        "target_mult": [1.0, 1.61, 2.0],
        "min_entry_time": ["00:00", "11:00"],
        "max_entry_time": [None, "13:00"],
    }
)
info = describe_space(DAY_SPACE, base=DAY_CONFIG)
print(f"search space: {info.size} combinations, recommended algorithm: {info.recommendation}")

day_opt = optimize(
    candles,
    DAY_CONFIG,
    DAY_SPACE,
    objective=OBJECTIVE,
    constraints=CONSTRAINTS,
    oos_fraction=0.3,
    top_n=5,
    timeframe=TIMEFRAME,
    intraday=INTRADAY,
)
print(f"{day_opt.n_trials} trials. best in-sample {OBJECTIVE}: {day_opt.best.objective:+.4f}")
print("best parameters:", day_opt.best.params)
day_opt.validation_table()

## 7. Next steps

- Genetic search behind the same `Optimizer` interface (brief, section 12), compared with the grid on a small space.
- 123 de venda, once its rules are written.
- An M30 or M15 export from MT5, to pinpoint the 16:30 exit with `day_exit_policy="auto"`.
- `score_v1` after its formula is approved in `docs/score.md`.